# 📓 Semana 1 · Dia 3 — Datasets do projeto e os 4 formatos de arquivo

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (formatos de dados) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Dados do projeto lidos em 4 formatos + inspeção |

---


## 📖 Teoria — O projeto do curso: vendas de varejo

Você é engenheiro(a) de dados de uma rede de varejo com múltiplas lojas. Os dados chegam em **CSV** (vendas), **JSON** (clientes) e feeds de catálogo. Nas próximas semanas você vai construir a plataforma completa: Bronze → Prata → Ouro, dashboards, um modelo de previsão, um RAG e um agente que responde perguntas sobre as vendas.

Usaremos o dataset oficial de exemplo do Databricks — **Online Retail** (~540 mil linhas de vendas de uma loja online do Reino Unido, dez/2010–dez/2011) — que já vem disponível no workspace (`samples.databricks.datasets` / `/databricks-datasets`), com espelho no GitHub oficial e na UCI (archive.ics.uci.edu/dataset/352/online+retail). Também geramos dados sintéticos de voos para enriquecimento.


## 📖 Teoria — Os 4 formatos que você precisa dominar

| Formato | Uso típico | Quando usar |
|---|---|---|
| **CSV** | Exportações, sistemas legados | Leitura rápida; sem schema |
| **JSON** | APIs, logs, eventos | Semianinhado, flexível |
| **Parquet** | Analítico, colunar, comprimido | Grande volume, schema forte |
| **Delta** | Tabelas transacionais (ACID) | **Sempre para produção** — base do Lakehouse |

**Parquet vs CSV**: Parquet é **colunar** (lê só as colunas necessárias), comprime 75–90% e guarda o **schema** no arquivo. CSV é linha a linha e sem schema.


### 💻 Na prática — Carregando o dataset

O Databricks já fornece o dataset **Online Retail** no workspace. Vamos usar, em ordem: (1) o dataset local do Databricks (sem internet, recomendado na Free Edition), (2) o raw do GitHub oficial do Databricks, (3) upload manual.

> 💾 **Destino dos arquivos**: usaremos o **Volume `vol_dados_curso`** (criado no Dia 2) — `/Volumes/workspace/bronze/vol_dados_curso/` — como área de trabalho do curso. Ele é governado pelo Unity Catalog e acessível em qualquer compute. Na Free Edition, o filesystem local (`/tmp`) é restrito, então **não** usamos `/tmp`.


In [ ]:
# 1) Dataset local do Databricks (funciona onde há sample datasets)
destino = "/Volumes/workspace/bronze/vol_dados_curso/vendas.csv"
caminhos = [
    "/databricks-datasets/online-retail-dataset/data-original/online-retail-dataset.csv",
    "/Volumes/samples/databricks/datasets/online_retail/online_retail.csv",
]
encontrado = None
for p in caminhos:
    try:
        dbutils.fs.head(p)  # só funciona se o ARQUIVO existir
        encontrado = p
        break
    except Exception:
        continue
if encontrado:
    dbutils.fs.cp(encontrado, destino)
    print("Usado dataset local do Databricks:", encontrado)
else:
    print("Dataset local não encontrado nesta conta — tente o GitHub (próxima célula).")

In [ ]:
# 2) Fallback: raw do GitHub oficial do Databricks (se internet liberada)
import urllib.request
url = "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"
try:
    with urllib.request.urlopen(url, timeout=60) as r:
        conteudo = r.read()
    # dbutils.fs.put grava direto no Volume a partir da memória —
    # sem usar o filesystem local (que é restrito na Free Edition)
    dbutils.fs.put(destino, conteudo.decode("utf-8"), overwrite=True)
    print("Download OK (GitHub oficial)! Tamanho:", len(conteudo), "bytes")
except Exception as e:
    print("Fallback GitHub falhou:", str(e)[:120])
    print("Siga o upload manual na próxima célula.")

In [ ]:
# 3) Verificação: o arquivo está no Volume?
try:
    dbutils.fs.head(destino)
    print("OK — vendas.csv pronto no Volume:", destino)
except Exception:
    print("ARQUIVO AINDA NÃO ESTÁ NO VOLUME. Faça o upload manual:")
    print("1. Baixe o CSV: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv")
    print("2. No Databricks: Data > Add Data > Upload File")
    print("3. Destino: Volumes > workspace > bronze > vol_dados_curso")
    print("4. Rode esta célula de novo para confirmar.")

In [ ]:
# Conferir o arquivo no Volume
display(dbutils.fs.ls("/Volumes/workspace/bronze/vol_dados_curso/"))

### 💻 Na prática — Leitura com Spark

O Spark infere o schema automaticamente. Sempre **confira o schema** antes de usar os dados — é a fonte de 80% dos bugs em pipelines.


In [ ]:
# Ler CSV com inferência de schema
df_vendas = (spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .load(destino))
df_vendas.printSchema()
df_vendas.show(5, truncate=False)
print("Total de linhas:", df_vendas.count())

### 💻 Na prática — Inspeção e estatísticas

`describe` dá estatísticas básicas (min/max/avg). Cuidado: strings viram stats estranhas; aplique em colunas numéricas.


In [ ]:
# Estatísticas descritivas
df_vendas.describe("Quantity", "UnitPrice").show()
# Contagem de nulos por coluna (base da qualidade de dados)
from pyspark.sql.functions import col, isnan, isnull, count
df_vendas.select([count(isnull(c)).alias(f"nulos_{c}") for c in df_vendas.columns]).show()

### 💻 Na prática — Gravando nos 4 formatos

Vamos gravar a mesma base em CSV (cópia), JSON, Parquet e Delta e comparar. Usamos o **Volume** como destino (área governada e persistente).


In [ ]:
# Gravar nos 4 formatos (no Volume do curso)
df_vendas.write.mode("overwrite").parquet("/Volumes/workspace/bronze/vol_dados_curso/vendas.parquet")
df_vendas.write.mode("overwrite").json("/Volumes/workspace/bronze/vol_dados_curso/vendas.json")
df_vendas.write.mode("overwrite").format("delta").save("/Volumes/workspace/bronze/vol_dados_curso/vendas_delta")
print("Parquet, JSON e Delta gravados no Volume vol_dados_curso")

In [ ]:
# Comparar leitura de volta e tamanho
df_p = spark.read.parquet("/Volumes/workspace/bronze/vol_dados_curso/vendas.parquet")
df_j = spark.read.json("/Volumes/workspace/bronze/vol_dados_curso/vendas.json")
df_d = spark.read.format("delta").load("/Volumes/workspace/bronze/vol_dados_curso/vendas_delta")
print("Parquet:", df_p.count(), "| JSON:", df_j.count(), "| Delta:", df_d.count())
display(spark.sql("SELECT COUNT(*) AS linhas FROM delta.`/Volumes/workspace/bronze/vol_dados_curso/vendas_delta`"))

> 🎯 **Dica de prova**: A prova DEA pergunta **quando usar cada formato** e o motivo de Parquet/Delta serem superiores ao CSV (colunar, compressão, schema embutido, ACID). Decore a tabela dos formatos.


## 🎯 Exercícios de fixação

**1.** Leia o JSON e mostre as 3 primeiras linhas.

**2.** Quantas linhas tem o JSON vs o Parquet? Por quê?

**3.** Crie uma view temporária `vendas_vw` a partir do DataFrame lido.

**4.** Qual formato escolheria para uma tabela de produção que recebe updates frequentes? Por quê?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Ler JSON

`spark.read.json('/Volumes/workspace/bronze/vol_dados_curso/vendas.json').show(3, truncate=False)`.

**2.** JSON vs Parquet

O JSON gravado aqui contém o mesmo conjunto de linhas (a menos que o parquet tenha partições extras); a diferença real é o tamanho em disco e a velocidade: Parquet comprime por coluna e guarda schema. Em pipelines reais o JSON é bem maior e mais lento.

**3.** View temporária

`df_vendas.createOrReplaceTempView('vendas_vw')` — a partir dela pode-se consultar com `%sql SELECT * FROM vendas_vw LIMIT 5`.

**4.** Delta

Delta: ACID, Time Travel, MERGE, schema evolution — necessário para updates frequentes e consistência. CSV/JSON não têm transações.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*